In [1]:
import os
import pandas as pd
cur_dir = os.getcwd()
raw_data_path = os.path.join("..", "data", "raw", "TEDS_Discharge.csv")
raw_df = pd.read_csv(raw_data_path)

In [2]:
len(raw_df.columns)

76

In [11]:
len(raw_df.columns) - 2 - 1 - 61

12

In [13]:
print("CASEID" in raw_df.columns)
print("DISYR" in raw_df.columns)
print("REASON" in raw_df.columns)

True
True
True


In [12]:
_d = []
_a = []
for col in raw_df.columns:
    if "_D" in col:
        _d.append(col)
        print(col)
    else:
        _a.append(col)
print(_d)
print(len(_d))
print(len(_a))

SERVICES_D
EMPLOY_D
LIVARAG_D
ARRESTS_D
DETNLF_D
SUB1_D
SUB2_D
SUB3_D
FREQ1_D
FREQ2_D
FREQ3_D
FREQ_ATND_SELF_HELP_D
['SERVICES_D', 'EMPLOY_D', 'LIVARAG_D', 'ARRESTS_D', 'DETNLF_D', 'SUB1_D', 'SUB2_D', 'SUB3_D', 'FREQ1_D', 'FREQ2_D', 'FREQ3_D', 'FREQ_ATND_SELF_HELP_D']
12
64


In [3]:
import pandas as pd
import numpy as np
import glob

def calculate_combined_stats(file_paths, sample_n=209121):
    """
    여러 CSV 파일을 읽어 변수(name)별 전체 평균과 표준편차를 계산합니다.
    """
    dfs = []
    for file in file_paths:
        df = pd.read_csv(file)
        # 필요한 컬럼만 추출 (변수명, 평균, 표준편차)
        dfs.append(df[['name', 'mean', 'std']])
    
    # 모든 데이터를 하나로 합침
    combined_df = pd.concat(dfs, ignore_index=True)
    
    results = []
    
    # 변수(name)별로 그룹화하여 계산
    for name, group in combined_df.groupby('name'):
        means = group['mean'].values
        stds = group['std'].values
        
        # 1. 전체 평균 (n이 동일하므로 단순히 평균의 평균)
        total_mean = np.mean(means)
        
        # 2. 전체 분산 계산
        # n이 동일하므로 공식: mean(std^2 + (group_mean - total_mean)^2)
        variances = stds ** 2
        diff_sq = (means - total_mean) ** 2
        total_variance = np.mean(variances + diff_sq)
        
        # 3. 전체 표준편차
        total_std = np.sqrt(total_variance)
        
        results.append({
            'name': name,
            'combined_mean': total_mean,
            'combined_std': total_std,
            # 'count_files': len(group)
        })
    
    return pd.DataFrame(results)

# --- 실행 부분 ---
# 파일 리스트 준비
files = ['all_folds_pi_seed1.csv', 'all_folds_pi_seed2.csv', 'all_folds_pi_seed3.csv']

# 함수 실행
final_df = calculate_combined_stats(files)

# 결과 확인 (평균이 높은 순서대로)
final_df = final_df.sort_values(by='combined_mean', ascending=False)
print(final_df.head(10))

# 결과를 CSV로 저장
final_df.to_csv('combined_results.csv', index=False)

                     name  combined_mean  combined_std
39                    LOS       0.094008      0.002050
62             SERVICES_D       0.055304      0.037093
63                 STFIPS       0.036279      0.010176
8                CBSA2020       0.035628      0.005418
61               SERVICES       0.034433      0.035073
21                FREQ1_D       0.032506      0.006246
38              LIVARAG_D       0.007383      0.000823
5               ARRESTS_D       0.007223      0.004317
27  FREQ_ATND_SELF_HELP_D       0.007082      0.001461
14               DIVISION       0.004609      0.003469
